# NB-01 · Experiment Setup
Generates: dataset statistics figures and configuration tables for Chapter 5.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'notebooks', 'chapter4'))
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from utils import setup_style, save_fig, load_results_history, ROOT, FIG_DIR

setup_style()
print('FIG_DIR:', FIG_DIR)

## 1. Dataset Statistics

In [ ]:
# Load a sample of eval_users to get click-sequence statistics
import random

eval_path = ROOT / 'evaluation' / 'eval_users.json'
with open(eval_path, encoding='utf-8') as f:
    all_users = json.load(f)

n_users = len(all_users)
train_lens = [len(u['train_clicks']) for u in all_users]
test_lens  = [len(u['test_clicks'])  for u in all_users]

print(f'Users          : {n_users:,}')
print(f'Train clicks   : mean={np.mean(train_lens):.1f}, median={np.median(train_lens):.0f}, '
      f'min={min(train_lens)}, max={max(train_lens)}')
print(f'Test clicks    : {set(test_lens)} (always 1)')

# Unique items in eval sequences
all_items = set()
for u in all_users:
    all_items.update(u['train_clicks'])
    all_items.update(u['test_clicks'])
print(f'Unique items in eval sequences: {len(all_items):,}')

In [ ]:
# Figure: Distribution of training sequence lengths
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Histogram of train_clicks lengths
ax = axes[0]
bins = [1, 5, 10, 20, 50, 100, 200, 500, 1000, max(train_lens) + 1]
counts, edges = np.histogram(train_lens, bins=bins)
labels = [f'{int(edges[i])}–{int(edges[i+1])-1}' for i in range(len(counts))]
labels[-1] = f'{int(edges[-2])}+'
bars = ax.bar(range(len(counts)), counts, color='#61AFEF', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9)
ax.set_xlabel('Training sequence length')
ax.set_ylabel('Number of users')
ax.set_title('(a) Distribution of training sequence lengths')
for bar, cnt in zip(bars, counts):
    if cnt > 1000:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{cnt/1000:.0f}k', ha='center', va='bottom', fontsize=8)

# Box plot / summary
ax2 = axes[1]
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
pct_values  = [int(np.percentile(train_lens, p)) for p in percentiles]
ax2.barh(range(len(percentiles)), pct_values, color='#98C379', edgecolor='white')
ax2.set_yticks(range(len(percentiles)))
ax2.set_yticklabels([f'p{p}' for p in percentiles])
ax2.set_xlabel('Sequence length (clicks)')
ax2.set_title('(b) Percentiles of training sequence length')
for i, v in enumerate(pct_values):
    ax2.text(v + 2, i, str(v), va='center', fontsize=9)

save_fig('fig_dataset_statistics', fig)
plt.show()

## 2. Evaluation Protocol Summary Table

In [ ]:
# Print evaluation protocol details for reference
protocol = {
    'Test users':          f'{n_users:,}',
    'Test items per user': '1 (leave-one-out)',
    'Negative sampling':   'Sampled N random negatives (N ∈ {99, 999})',
    'Primary protocol':    'N = 99 (SASRec/BERT4Rec standard)',
    'Random baseline HR@10': '0.100 (10/100 items)',
    'Metrics':             'HR@5, HR@10, NDCG@10, MRR@10',
    'Evaluation seed':     '42 (canonical); multi-seed: 42/123/456/789/2026',
}
print('=== Evaluation Protocol ===')
for k, v in protocol.items():
    print(f'  {k:30s}: {v}')

## 3. Model Configuration Table

In [ ]:
dif_sasrec_config = {
    'Architecture':         'DIF-SASRec (Decoupled Intent-Feature Self-Attentive Rec.)',
    'Attention blocks':     '4',
    'Attention heads':      '8',
    'Hidden dimension':     '512',
    'Max sequence length':  '200',
    'Parameters':           '~12.4M',
    'Training users':       '100,000',
    'Epochs':               '30 (pretrained)',
    'Batch size':           '2048',
    'Optimizer':            'AdamW',
    'LR schedule':          'Cosine annealing',
    'Encoder (text)':       'BGE-M3 (BAAI/bge-m3), 1024-dim, fp16, CUDA',
    'Content veto τ':       '0.3 (cosine similarity threshold)',
}

pipeline_a_config = {
    'Graph embedding':      'Cleora (co-purchase behavioral graph)',
    'Cleora index size':    '375,280 items',
    'Profile encoder':      'BGE-M3 (mean of clicked item embeddings)',
    'FAISS index (prod)':   'HNSW, 1.7M vectors, 1024-dim',
    'FAISS index (eval)':   'Flat, 3M vectors, exact search',
    'Candidate pool':       'Top-375k from Cleora, re-ranked by cosine',
    'RRF constant k':       '60 (for A+B fusion)',
}

print('=== DIF-SASRec Configuration ===')
for k, v in dif_sasrec_config.items():
    print(f'  {k:25s}: {v}')

print('\n=== Pipeline A Configuration ===')
for k, v in pipeline_a_config.items():
    print(f'  {k:25s}: {v}')

## 4. Figure: Evaluation Protocol Schematic

In [ ]:
# Visualise the sampled-evaluation protocol
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis('off')

# User history
ax.text(0.3, 3.5, 'User history', fontsize=10, fontweight='bold')
for i in range(8):
    color = '#61AFEF' if i < 7 else '#98C379'
    rect = mpatches.FancyBboxPatch((0.3 + i * 0.55, 2.8), 0.45, 0.5,
                                   boxstyle='round,pad=0.05', facecolor=color, edgecolor='white')
    ax.add_patch(rect)
    lbl = 'Test' if i == 7 else f'i{i+1}'
    ax.text(0.3 + i * 0.55 + 0.225, 3.05, lbl, ha='center', va='center', fontsize=8,
            color='white', fontweight='bold')

# Arrow
ax.annotate('', xy=(5.0, 1.9), xytext=(5.0, 2.7),
            arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Candidate pool
ax.text(0.3, 1.75, 'Candidate pool (N+1 items)', fontsize=10, fontweight='bold')
rect_test = mpatches.FancyBboxPatch((0.3, 1.0), 0.7, 0.6,
                                    boxstyle='round,pad=0.05', facecolor='#98C379', edgecolor='white')
ax.add_patch(rect_test)
ax.text(0.65, 1.3, 'Test', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

for j in range(8):
    rect_neg = mpatches.FancyBboxPatch((1.2 + j * 0.85, 1.0), 0.75, 0.6,
                                       boxstyle='round,pad=0.05', facecolor='#9E9E9E', edgecolor='white', alpha=0.7)
    ax.add_patch(rect_neg)
ax.text(5.5, 1.3, 'N random negatives', ha='center', va='center', fontsize=9, color='white')
ax.text(9.0, 1.3, '...', ha='center', va='center', fontsize=14, color='#555')

# Metrics
ax.text(0.3, 0.55, 'Metrics: HR@K  ·  NDCG@K  ·  MRR@K', fontsize=10)
ax.text(0.3, 0.15, 'N = 99 (primary)  or  N = 999 (strict)',
        fontsize=9, color='#555', style='italic')

save_fig('fig_eval_protocol', fig)
plt.show()